In [3]:
import time
import random

FILE_NAME = "server_logs.txt"
TOTAL_LINES = 1_000_000

def generate_huge_log():
    statuses = ["INFO: All systems operational.", "WARN: CPU load is 75%.", "DEBUG: Connection established."]
    
    with open(FILE_NAME, "w") as f:
        for i in range(TOTAL_LINES):
            if i == 950_000:
                f.write(f"[{i}] FATAL: database_melted_down_completely\n")
            else:
                f.write(f"[{i}] {random.choice(statuses)}\n")

def test_full_table_scan():
    start_time = time.time()
    
    found_lines = []
    with open(FILE_NAME, "r") as f:
        for line in f:
            if "melted" in line:
                found_lines.append(line.strip())
                
    elapsed = time.time() - start_time
    print(f"Найдено: {found_lines}")
    print(f"Время поиска: {elapsed:.4f} секунд\n")
    return elapsed

def test_inverted_index():   

    inverted_index = {}
    with open(FILE_NAME, "r") as f:
        for line_num, line in enumerate(f):
            # Если видим критическое слово, сохраняем номер строки
            if "melted" in line:
                if "melted" not in inverted_index:
                    inverted_index["melted"] = []
                inverted_index["melted"].append(line_num)
                
    start_time = time.time()
    
    result_line_numbers = inverted_index.get("melted", [])
    
    elapsed = time.time() - start_time
    print(f"Найдено в строках: {result_line_numbers}")
    print(f"Время поиска: {elapsed:.6f} секунд (Внимание на нули!)")
    return elapsed

if __name__ == "__main__":
    generate_huge_log()
    t1 = test_full_table_scan()
    t2 = test_inverted_index()
    
    print("-" * 40)
    print(f"ИТОГ: Инвертированный индекс быстрее в {t1 / t2:,.0f} раз!")


Найдено: ['[950000] FATAL: database_melted_down_completely']
Время поиска: 0.0489 секунд

Найдено в строках: [950000]
Время поиска: 0.000000 секунд (Внимание на нули!)
----------------------------------------


ZeroDivisionError: float division by zero